# Left-wing votes per municipality

This notebook calculates the percentage of valid votes for two groups of parties in the 2025 Dutch parliamentary election.

**Input (`data/raw/`):** `tk2025_uitslag.csv` — election results per municipality and party.

**Output (`data/processed/`):** `left_wing_votes_per_municipality.csv` — one row per municipality (342).

- **Broad left-wing vote:** GroenLinks–PvdA, SP, Partij voor de Dieren, DENK, BIJ1, D66, Volt.
- **Narrow left-wing vote:** GroenLinks–PvdA, SP, Partij voor de Dieren, DENK, BIJ1.

The party groupings are operational definitions for this project, not official classifications.

In [ ]:
import pandas as pd

In [ ]:
# Load election results
election = pd.read_csv(
    "../data/raw/tk2025_uitslag.csv",
    sep=";",
    encoding="utf-8-sig"
)

# Define the two party groups
narrow_parties = [
    "GROENLINKS / Partij van de Arbeid (PvdA)",
    "SP (Socialistische Partij)",
    "Partij voor de Dieren",
    "DENK",
    "BIJ1"
]
broad_parties = narrow_parties + ["D66", "Volt"]

# Keep party-level votes in municipalities (exclude national and NBSB totals)
municipality_rows = election["RegioCode"].str.match(r"^G\d+$", na=False)
party_votes = election.loc[
    municipality_rows & (election["VeldType"] == "LijstAantalStemmen") &
    (election["Regio"] != "NBSB"),
    ["Regio", "LijstNaam", "Waarde"]
].copy()

# Valid votes are the denominator, excluding blank and invalid ballots
valid_votes = election.loc[
    municipality_rows & (election["VeldType"] == "AantalGeldigeStemmen") &
    (election["Regio"] != "NBSB"),
    ["Regio", "Waarde"]
].rename(columns={"Waarde": "valid_votes"})

# Sum votes for each party group
wide = party_votes.pivot_table(
    index="Regio", columns="LijstNaam", values="Waarde",
    aggfunc="sum", fill_value=0
)
assert set(broad_parties).issubset(wide.columns), "One or more party names are missing"
result = pd.DataFrame({
    "Regio": wide.index,
    "broad_votes": wide[broad_parties].sum(axis=1).to_numpy(),
    "narrow_votes": wide[narrow_parties].sum(axis=1).to_numpy()
})
result = result.merge(valid_votes, on="Regio", how="inner", validate="one_to_one")

# Match the municipality names used in the other processed files
name_mapping = {
    "'s-Gravenhage": "Den Haag",
    "Hengelo (O)": "Hengelo",
    "Bergen (L)": "Bergen (L.)",
    "Bergen (NH)": "Bergen (NH.)"
}
result["municipality"] = result["Regio"].replace(name_mapping)

# Calculate vote percentages
result["left_wing_vote_pct"] = (100 * result["broad_votes"] / result["valid_votes"]).round(2)
result["left_wing_vote_narrow_pct"] = (100 * result["narrow_votes"] / result["valid_votes"]).round(2)

# Keep the existing output filename and column names
data = result[[
    "municipality", "left_wing_vote_pct", "left_wing_vote_narrow_pct"
]].copy()

assert data.shape == (342, 3), f"Unexpected shape: {data.shape}"
assert data["municipality"].is_unique, "Duplicate municipality names"
assert not data.isna().any().any(), "Missing values found"
print(data.shape)

# Save
data.to_csv("../data/processed/left_wing_votes_per_municipality.csv", index=False)

In [ ]:
data.head()